# Get Embedding from Molecular Represantation Methods

In [2]:
import pickle, h5py
import numpy as np
import pandas as pd

# read SMILES from CIGS/CIGS_Smiles_list.csv

smiles_df = pd.read_csv('CIGS/CIGS_Smiles_list.csv')
smiles_list = smiles_df['SMILES'].tolist()

In [4]:
print(len(smiles_list), smiles_list[0])

10726 B(C(CC(C)C)NC(=O)C(C(C)O)NC(=O)C1=CC=CC(=N1)C2=CC=CC=C2)(O)O


In [ ]:
# 检查是否已经去过重
smiles_list = list(set(smiles_list))
print(len(smiles_list), smiles_list[0])

10726 CC(C)(C1=CC=C(C=C1)O)C2=CC=C(C=C2)O


# Generate ECFP4 embedding for all SMILES


In [6]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import pickle

def smiles_to_ecfp4(smiles_list, radius=2, n_bits=2048):
    """
    将SMILES列表转换为ECFP4特征向量。

    参数:
    smiles_list (list): SMILES字符串列表。
    radius (int): ECFP的半径，默认为2（即ECFP4）。
    n_bits (int): 特征向量的长度，默认为2048。

    返回:
    dict: 一个字典，键为SMILES字符串，值为对应的ECFP4特征向量。
    """
    ecfp4_dict = {}
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            # 替换编码器
            ecfp4 = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
            ecfp4_dict[smiles] = np.array(ecfp4, dtype=np.float32)
        else:
            print("Error in ", smiles)
            # 如果SMILES无法解析为分子，则填充一个全零的向量
            ecfp4_dict[smiles] = np.zeros(n_bits, dtype=np.float32)
    
    return ecfp4_dict

# 计算ECFP4特征
ecfp4_features_dict = smiles_to_ecfp4(smiles_list)

print(len(smiles_list), len(ecfp4_features_dict))

[21:26:04] WARNING: not removing hydrogen atom without neighbors
[21:26:04] WARNING: not removing hydrogen atom without neighbors
[21:26:04] WARNING: not removing hydrogen atom without neighbors


10726 10726


In [7]:
# 保存为 pickle 文件
with open('./CIGS/ECFP4_emb2048.pickle', 'wb') as f:
    pickle.dump(ecfp4_features_dict, f)

# 3D  UniMol

In [ ]:
import os
import pickle
import numpy as np
from tqdm import tqdm
from unimol_tools import UniMolRepr
import os, sys, contextlib

@contextlib.contextmanager
def suppress_stdout_stderr():
    """with 块内所有 print / tqdm / warning 都不会显示"""
    with open(os.devnull, 'w') as devnull:
        old_out, old_err = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = devnull, devnull
        try:
            yield
        finally:
            sys.stdout, sys.stderr = old_out, old_err

def get_unimol_embeddings(smiles_list, output_file="UniMol_emb512.pkl", 
                         model_name='unimolv1', model_size='84m', 
                         remove_hs=False, batch_size=32):
    """
    使用Uni-Mol模型为SMILES列表生成分子嵌入，并保存为pickle文件
    
    参数:
    smiles_list (list): SMILES字符串列表
    output_file (str): 输出pickle文件路径
    model_name (str): 模型名称，可选'unimolv1'或'unimolv2'
    model_size (str): 模型大小，仅在使用unimolv2时有效
    remove_hs (bool): 是否移除氢原子
    batch_size (int): 批处理大小
    
    返回:
    dict: 包含SMILES及其对应嵌入的字典
    """
    # 初始化模型
    clf = UniMolRepr(
        data_type='molecule',
        remove_hs=remove_hs,
        model_name=model_name,
        model_size=model_size
    )
    
    # 用于存储结果的字典
    embeddings_dict = {}
    error_smiles = []
    
    # 批处理SMILES
    total_batches = (len(smiles_list) + batch_size - 1) // batch_size
    
    print(f"开始生成{len(smiles_list)}个SMILES的嵌入表示...")
    
    print("开始")
    # with suppress_stdout_stderr():
    for i in tqdm(range(total_batches), desc="处理批次"):
        batch = smiles_list[i*batch_size : (i+1)*batch_size]
        
        try:
            # 获取嵌入表示
            batch_repr = clf.get_repr(batch, return_atomic_reprs=False)
            
            # 将结果存入字典 (使用CLS token作为分子表示)
            for idx, smiles in enumerate(batch):
                embeddings_dict[smiles] = batch_repr['cls_repr'][idx]
                
        except Exception as e:
            print(f"处理批次 {i+1}/{total_batches} 时发生错误: {str(e)}")
            # 记录处理失败的SMILES
            error_smiles.extend(batch)
        
    # 保存嵌入结果
    with open(output_file, 'wb') as f:
        pickle.dump(embeddings_dict, f)
    
    print(f"嵌入生成完成！共处理 {len(smiles_list)} 个SMILES，"
          f"{len(smiles_list) - len(error_smiles)} 个成功，"
          f"{len(error_smiles)} 个失败。")
    print(f"嵌入结果已保存至 {output_file}")
    
    if error_smiles:
        print(f"处理失败的SMILES已记录。")
    
    return embeddings_dict

# 使用示例
if __name__ == "__main__":
    
    # UniMol V1
    # embeddings = get_unimol_embeddings(
    #     smiles_list=smiles_list,
    #     output_file="embeddings/UniMol_emb512.pkl",
    #     model_name='unimolv1',
    #     model_size='84m',
    #     remove_hs=False,
    #     batch_size=32
    # )
    
    # UniMol V2
    embeddings = get_unimol_embeddings(
    smiles_list=smiles_list,
    output_file="embeddings/UniMolV2_emb1024.pkl", # save path
    model_name='unimolv2',      # 修改为 v2 版本
    model_size='310m',          # 指定 v2 模型大小（根据需要选择）
    remove_hs=False,
    batch_size=32
    )